In [2]:
import os
import gzip
from collections import Counter
import ijson  # pip install ijson
import pandas as pd
from pathlib import Path

FILES = {
    'nif_context_en.ttl': 'nif_context_en.ttl',      # ~28G - training corpus
    'labels_en.ttl': 'labels_en.ttl',                # ~900M - 4.8M entities
    'instance_types_en.ttl': 'instance_types_en.ttl', # ~220M - typed entities
    'redirects_en.ttl': 'redirects_en.ttl'           # ~500M - normalization
}

base_dir = Path('../dbpedia-2016-10')
for f, path in FILES.items():
    full_path = base_dir / path
    print(f"\n{f}")
    print(f"size: {full_path.stat().st_size / 1e9:.2f} GB")
    print(f"lines: {sum(1 for _ in open(full_path)):,}")


nif_context_en.ttl
size: 20.41 GB
lines: 29,642,397

labels_en.ttl
size: 1.59 GB
lines: 12,845,254

instance_types_en.ttl
size: 0.75 GB
lines: 5,150,434

redirects_en.ttl
size: 1.18 GB
lines: 7,699,478


In [3]:
for f, path in FILES.items():
    print(f"\n {path}")
    with open(base_dir / path) as fp:
        for i, line in enumerate(fp):
            print(repr(line.rstrip()))
            if i >= 9: break


 nif_context_en.ttl
'# started 2017-02-07T14:14:05Z'
'<http://dbpedia.org/resource/Animalia_(book)?dbpv=2016-10&nif=context> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#Context> .'
'<http://dbpedia.org/resource/Animalia_(book)?dbpv=2016-10&nif=context> <http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#beginIndex> "0"^^<http://www.w3.org/2001/XMLSchema#nonNegativeInteger> .'
'<http://dbpedia.org/resource/Animalia_(book)?dbpv=2016-10&nif=context> <http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#endIndex> "2294"^^<http://www.w3.org/2001/XMLSchema#nonNegativeInteger> .'
'<http://dbpedia.org/resource/Animalia_(book)?dbpv=2016-10&nif=context> <http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#sourceUrl> <http://en.wikipedia.org/wiki/Animalia_(book)?oldid=741600610> .'
'<http://dbpedia.org/resource/Animalia_(book)?dbpv=2016-10&nif=context> <http://persistence.uni-leipzig.org/nlp2rd

In [4]:
def parse_ttl_sample(file_path, max_lines=1000):
    triples = []
    with open(file_path) as f:
        for i, line in enumerate(f):
            if i >= max_lines: break
            line = line.strip()
            if line and not line.startswith('#'):
                parts = [p.strip('<> "') for p in line.split() if p.strip()]
                if len(parts) >= 3:
                    triples.append((parts[0], parts[1], parts[2]))
    return pd.DataFrame(triples, columns=['subject', 'predicate', 'object'])

for f, path in FILES.items():
    df = parse_ttl_sample(base_dir / path)
    print(f"\n## {f} Structure (sample {len(df)} triples)")
    print(df['predicate'].value_counts().head())
    print("\nSample subjects:", df['subject'].str.startswith('http://dbpedia.org/resource/').sum(), f"/ {len(df)}")
    display(df.head(3))


## nif_context_en.ttl Structure (sample 999 triples)
predicate
http://www.w3.org/1999/02/22-rdf-syntax-ns#type                              167
http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#beginIndex    167
http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#predLang      167
http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#endIndex      166
http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#sourceUrl     166
Name: count, dtype: int64

Sample subjects: 999 / 999


,subject,predicate,object
0,http://dbpedia.org/resource/Animalia_(book)?db...,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://persistence.uni-leipzig.org/nlp2rdf/ont...
1,http://dbpedia.org/resource/Animalia_(book)?db...,http://persistence.uni-leipzig.org/nlp2rdf/ont...,"0""^^<http://www.w3.org/2001/XMLSchema#nonNegat..."
2,http://dbpedia.org/resource/Animalia_(book)?db...,http://persistence.uni-leipzig.org/nlp2rdf/ont...,"2294""^^<http://www.w3.org/2001/XMLSchema#nonNe..."



## labels_en.ttl Structure (sample 999 triples)
predicate
http://www.w3.org/2000/01/rdf-schema#label    999
Name: count, dtype: int64

Sample subjects: 999 / 999


,subject,predicate,object
0,http://dbpedia.org/resource/AccessibleComputing,http://www.w3.org/2000/01/rdf-schema#label,"AccessibleComputing""@en"
1,http://dbpedia.org/resource/AfghanistanHistory,http://www.w3.org/2000/01/rdf-schema#label,"AfghanistanHistory""@en"
2,http://dbpedia.org/resource/AfghanistanGeography,http://www.w3.org/2000/01/rdf-schema#label,"AfghanistanGeography""@en"



## instance_types_en.ttl Structure (sample 999 triples)
predicate
http://www.w3.org/1999/02/22-rdf-syntax-ns#type    999
Name: count, dtype: int64

Sample subjects: 999 / 999


,subject,predicate,object
0,http://dbpedia.org/resource/Anarchism,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2002/07/owl#Thing
1,http://dbpedia.org/resource/Achilles,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2002/07/owl#Thing
2,http://dbpedia.org/resource/Autism,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://dbpedia.org/ontology/Disease



## redirects_en.ttl Structure (sample 999 triples)
predicate
http://dbpedia.org/ontology/wikiPageRedirects    999
Name: count, dtype: int64

Sample subjects: 999 / 999


,subject,predicate,object
0,http://dbpedia.org/resource/AccessibleComputing,http://dbpedia.org/ontology/wikiPageRedirects,http://dbpedia.org/resource/Computer_accessibi...
1,http://dbpedia.org/resource/AfghanistanHistory,http://dbpedia.org/ontology/wikiPageRedirects,http://dbpedia.org/resource/History_of_Afghani...
2,http://dbpedia.org/resource/AfghanistanGeography,http://dbpedia.org/ontology/wikiPageRedirects,http://dbpedia.org/resource/Geography_of_Afgha...


In [5]:
import re

def count_dbpedia_resources(file_path):
    """Count unique http://dbpedia.org/resource/* subjects/objects"""
    resources = set()
    with open(file_path) as f:
        content = f.read()  # or stream for huge files
        matches = re.findall(r'http://dbpedia\.org/resource/[^<>\s"]+', content)
        return len(set(matches))

# SAFE streaming version (memory-efficient)
def stream_count_resources(file_path):
    resources = Counter()
    with open(file_path) as f:
        buffer = ''
        for line in f:
            buffer += line
            while '\n' in buffer:
                triple, buffer = buffer.split('\n', 1)
                if triple.strip() and not triple.startswith('#'):
                    matches = re.findall(r'http://dbpedia\.org/resource/[^<>\s"]+', triple)
                    for m in matches:
                        resources[m] += 1
    return len(resources)

print("LABELS (4.8M expected):")
labels_count = stream_count_resources(base_dir / 'labels_en.ttl')
print(f"Unique DBpedia resources: {labels_count:,}")

print("\nINSTANCE_TYPES (~4M expected):")
inst_count = stream_count_resources(base_dir / 'instance_types_en.ttl')
print(f"Unique DBpedia resources: {inst_count:,}")

LABELS (4.8M expected):
Unique DBpedia resources: 12,845,247

INSTANCE_TYPES (~4M expected):
Unique DBpedia resources: 5,044,223


In [ ]:
#!pip install gensim==3.4.0 doesnt work so we keep 4.4

import gensim
from gensim.models import Word2Vec
import re
import ijson
from pathlib import Path
from tqdm import tqdm
import pickle

nif_path = Path('nif_context_en.ttl')
sentences_file = 'sentences.pkl'  # Save tokenized corpus
model_file = 'joint_embeddings_200d.model'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 8.7 MB/s eta 0:00:0000:0100:01m
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [43 lines of output]
      Extracting in /tmp/tmp5xuxhg9y
      Traceback (most recent call last):
        File "/tmp/pip-install-8e_6k0ih/gensim_65fb7d2ca28d482181ea316b59728578/ez_setup.py", line 136, in use_setuptools
          import pkg_resources
      ModuleNotFoundError: No module named 'pkg_resources'
      
      During handling of the above exception, another exception occurred:
      
      Traceback (most recent call last):
        File "/home/alant/myglobal/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 353, in <module>
          main()
        File "/home/alant/myglobal/lib/python3.12/site-packages/pip/_vendor/pyproject_hook